First part unzips all image folders and saves into the level above

In [ ]:
import zipfile
from pathlib import Path

source_folder = Path(r"E:\AUV_kathy\Mission_59_20260709_1\CathxC")
dest_folder = source_folder

for zip_path in source_folder.glob("*.zip"):
    print(f"Extracting {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_folder)

print("Done.")

The next part extracts the metadata from the images and saves in CSV

In [2]:
import os
import re
import csv
import xml.etree.ElementTree as ET

# ==========================================================
# User settings
# ==========================================================

basepath = r"E:\AUV_kathy"
mission_name = "Mission_59_20260709_1"

input_folder = os.path.join(basepath, mission_name)

output_dir = r"C:\Users\a39799\OneDrive - Danmarks Tekniske Universitet\Documents\AUV"
os.makedirs(output_dir, exist_ok=True)

output_csv = os.path.join(output_dir, f"{mission_name}_image_metadata.csv")

# ==========================================================

# Find XML metadata embedded in image
xml_pattern = re.compile(rb"<\?xml.*?</image>", re.DOTALL)

fields = [
    "filename",
    "image_time", "image_date", "acq_index",
    "lat", "long", "altitude", "depth",
    "pitch", "roll", "yaw",
    "exposure", "digital_gain", "analog_gain", "sensor_gain",
    "aperture", "focus", "camera_name", "session_name",
    "serial_number"
]

rows = []

image_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".jfif"))
]

print(f"Found {len(image_files)} images")

for filename in image_files:

    filepath = os.path.join(input_folder, filename)

    with open(filepath, "rb") as f:
        data = f.read()

    match = xml_pattern.search(data)

    if not match:
        print(f"No XML found in {filename}")
        continue

    try:
        root = ET.fromstring(match.group().decode("utf-8", errors="ignore"))
    except ET.ParseError:
        print(f"Could not parse XML in {filename}")
        continue

    coords = root.find("./Position/Coords")
    depth = root.find("./Position/Depth")
    direction = root.find("./Position/Direction")
    acquisition = root.find("./acquisition")
    versions = root.find("./versions")

    rows.append({
        "filename": filename,
        "image_time": root.attrib.get("time", ""),
        "image_date": root.attrib.get("date", ""),
        "acq_index": root.attrib.get("acq_index", ""),
        "lat": coords.attrib.get("lat", "") if coords is not None else "",
        "long": coords.attrib.get("long", "") if coords is not None else "",
        "altitude": depth.attrib.get("altitude", "") if depth is not None else "",
        "depth": depth.attrib.get("depth", "") if depth is not None else "",
        "pitch": direction.attrib.get("pitch", "") if direction is not None else "",
        "roll": direction.attrib.get("roll", "") if direction is not None else "",
        "yaw": direction.attrib.get("yaw", "") if direction is not None else "",
        "exposure": acquisition.findtext("exposure", "") if acquisition is not None else "",
        "digital_gain": acquisition.findtext("digital_gain", "") if acquisition is not None else "",
        "analog_gain": acquisition.findtext("analog_gain", "") if acquisition is not None else "",
        "sensor_gain": acquisition.findtext("sensor_gain", "") if acquisition is not None else "",
        "aperture": acquisition.findtext("aperture", "") if acquisition is not None else "",
        "focus": acquisition.findtext("focus", "") if acquisition is not None else "",
        "camera_name": acquisition.findtext("name", "") if acquisition is not None else "",
        "session_name": acquisition.findtext("camera_session_name", "") if acquisition is not None else "",
        "serial_number": versions.findtext("serial_number", "") if versions is not None else "",
    })

with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

print(f"\nExtracted metadata from {len(rows)} images")
print(f"Saved to:\n{output_csv}")

Found 34567 images

Extracted metadata from 34567 images
Saved to:
C:\Users\a39799\OneDrive - Danmarks Tekniske Universitet\Documents\AUV\Mission_59_20260709_1_image_metadata.csv
